In [3]:
import math
bytes_per_weight = 4




In [4]:
def transformer_mem_and_flop(vocab_size, context_length, num_layers, d_model, num_heads, d_ff = None):
    if d_ff == None:
        d_ff = math.ceil(d_model * (8/3) * (1/64)) * 64

    params_embedding = vocab_size * d_model
    params_per_MHA = (d_model**2) * 4 # W_Q, W_K, W_V, W_O
    params_MHA = params_per_MHA * num_layers 
    params_per_FF = d_model * d_ff * 3
    params_FF = params_per_FF * num_layers 
    params_RMS_per_head = d_model * 2
    params_RMS = params_RMS_per_head * num_layers + d_model
    params_final_lin = d_model * vocab_size
    params_total = params_RMS + params_MHA + params_FF + params_final_lin + params_embedding

    muls_per_MHA = 4 * d_model**2 * context_length + 2 * d_model * context_length**2
    muls_per_FF = d_ff * d_model * context_length * 3
    muls_final_layer = context_length * d_model * vocab_size
    total_muls = (muls_per_FF+muls_per_MHA) * num_layers + muls_final_layer
    total_flops = total_muls * 2

    gflop_total = total_flops / 1e9
    gflop_MHA_attention = 2 * 2 * d_model * context_length**2 * num_layers / 1e9
    gflop_MHA_other = 2 * 4 * d_model**2 * context_length * num_layers / 1e9
    gflop_FF = d_ff * d_model * context_length * 3 * num_layers * 2 / 1e9
    gflop_final_lin = context_length * d_model * vocab_size * 2 / 1e9


    
    print(f"vocab_size: {vocab_size}\n"
      f"context_length: {context_length}\n"
      f"num_layers: {num_layers}\n"
      f"d_model: {d_model}\n"
      f"num_heads: {num_heads}\n"
      f"d_ff: {d_ff}\n\n"
      f"total params: {params_total:.3e}\n"
      f"    params embedding: {params_embedding:.3e}, {params_embedding/params_total:.2%}\n"
      f"    params MHA: {params_MHA:.3e}, {params_MHA/params_total:.2%}\n"
      f"    params FF: {params_FF:.3e}, {params_FF/params_total:.2%}\n"
      f"    params RMS: {params_RMS:.3e}, {params_RMS/params_total:.2%}\n"
      f"    params final lin: {params_final_lin:.3e}, {params_final_lin/params_total:.2%}\n"
      f"total gflop/forward pass: {gflop_total : .2f}\n"
      f"    MHA attention (d_model * context_len^2 * stuff): {gflop_MHA_attention : .2f}, {gflop_MHA_attention/gflop_total:.2%} \n"
      f"    MHA other (d_model^2 * context_len * stuff): {gflop_MHA_other:.2f}, {gflop_MHA_other/gflop_total:.2%} \n"
      f"    FF: {gflop_FF:.2f}, {gflop_FF/gflop_total:.2%}\n"
      f"    Final layer: {gflop_final_lin:.2f}, {gflop_final_lin/gflop_total:.2%}\n")

In [5]:
transformer_mem_and_flop(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25)

print("\nGPT2 Small:")
transformer_mem_and_flop(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 12, 
            d_model = 768, 
            num_heads = 12)

print("\nGPT2 Medium:")
transformer_mem_and_flop(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 24, 
            d_model = 1024, 
            num_heads = 16)

print("\nGPT2 Large:")
transformer_mem_and_flop(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 36, 
            d_model = 1280, 
            num_heads = 20)


print("\nGPT2 XL")
transformer_mem_and_flop(vocab_size = 50257, 
            context_length = 1024, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25)

print("\nGPT2 XL big context")
transformer_mem_and_flop(vocab_size = 50257, 
            context_length = 16384, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25)

print("\nGPT2 10B ")
transformer_mem_and_flop(vocab_size = 10000, 
            context_length = 512, 
            num_layers = 50, 
            d_model = 4608, 
            num_heads = 36)



vocab_size: 50257
context_length: 1024
num_layers: 48
d_model: 1600
num_heads: 25
d_ff: 4288

total params: 1.640e+09
    params embedding: 8.041e+07, 4.90%
    params MHA: 4.915e+08, 29.96%
    params FF: 9.880e+08, 60.22%
    params RMS: 1.552e+05, 0.01%
    params final lin: 8.041e+07, 4.90%
total gflop/forward pass:  3516.77
    MHA attention (d_model * context_len^2 * stuff):  322.12, 9.16% 
    MHA other (d_model^2 * context_len * stuff): 1006.63, 28.62% 
    FF: 2023.33, 57.53%
    Final layer: 164.68, 4.68%


GPT2 Small:
vocab_size: 50257
context_length: 1024
num_layers: 12
d_model: 768
num_heads: 12
d_ff: 2048

total params: 1.621e+08
    params embedding: 3.860e+07, 23.80%
    params MHA: 2.831e+07, 17.46%
    params FF: 5.662e+07, 34.92%
    params RMS: 1.920e+04, 0.01%
    params final lin: 3.860e+07, 23.80%
total gflop/forward pass:  291.65
    MHA attention (d_model * context_len^2 * stuff):  38.65, 13.25% 
    MHA other (d_model^2 * context_len * stuff): 57.98, 19.88% 
 

In [11]:
from torchgen import context


def adamw_accounting(vocab_size, context_length, num_layers, d_model, num_heads, batch_size, d_ff = None):
    # W_Q, W_K, W_V, W_O contribute d_model^2 each per layer
    # FF contributes 3 * d_model * d_ff = 8 d_model^2 per layer using assumption d_ff = (8/3) d_model
    # ignore RMS here, lower order
    # embedding + FF use d_model * vocab_size
    total_params = 12 * d_model**2 * num_layers + 2 * d_model * vocab_size

    # we do not count input in mem but do count output
    # (context_length * d_model + # RMS
        # 3 * context_length * d_model + # Q,K,V
        # 2 * context_length**2 * num_heads + # QK^T and softmax(QK^T)
        # context_length * d_model + # out values
        # context_length * d_model + # W_O of out
        # context_length * d_model + # RMS of out
        # context_length * (8/3) * d_model * 4 + # W1 x, W3x, swilu(W1x), swilu(W1x) * W3x
        # context_length * d_model # output of FF
        # ) = 2 * context_length**2 + context_length * d_model * (1 + 3 + 1 + 1 + 10 + (2/3) + 1 = 18 +2/3)
    per_blockbatch_activations = 2 * context_length**2 * num_heads + (18 + 2/3) * context_length * d_model


    total_activations_per_batch = (per_blockbatch_activations * num_layers + # transformer blocks
                         context_length * d_model * 2 + # initial embedding + final RMS 
                         2 * context_length * vocab_size) # output logits + softmax

    scalars_adam3 = (total_params * 4 + # param, 2 moments, gradient 
                    total_activations_per_batch * batch_size)


    # P = 12 * d_model**2 * num_layers + 2 * d_model * vocab_size
    # A = 2 * context_length**2 * num_heads * num_layers + (num_layers * 18.66 + 2) * context_length * d_model + 2 * context_length * vocab_size
    # max bytes = 4 * (4P + A*batch_size) 
    # outer 4 is floats -> bytes
    # inner 4 is params, gradiensts, and 2 moment buffers
    bytes_adamw = 4 * scalars_adam3
    print(f"vocab_size: {vocab_size}\n"
        f"context_length: {context_length}\n"
        f"num_layers: {num_layers}\n"
        f"d_model: {d_model}\n"
        f"num_heads: {num_heads}\n"
        f"batch_size: {batch_size}\n"
        f"total gb AdamW = {bytes_adamw / 1e9 :.2f}gb = \n"
        f"  (internal weights) 192 * d_model**2 * num_layers: {192 * d_model**2 * num_layers/1e9:.2f}gb, {192 * d_model**2 * num_layers/bytes_adamw:.2%}\n"
        f"  (embedding weights) 32 * d_model * vocab_size: {32 * d_model * vocab_size/1e9:.2f}gb, {32 * d_model * vocab_size/bytes_adamw:.2%}\n"
        f"  (logit type activations) 8 * context_length * vocab_size * batch_size: {8 * context_length * vocab_size * batch_size/1e9:.2f}gb, {8 * context_length * vocab_size * batch_size/bytes_adamw:.2%}\n"
        f"  (attention activations)  8 * context_length**2 * num_heads * num_layers * batch_size: {8 * context_length**2 * num_heads * num_layers * batch_size/1e9:.2f}gb, {8 * context_length**2 * num_heads * num_layers * batch_size/bytes_adamw:.2%}\n"
        f"  (internal activations) (74.666 * num_layers + 8) * context_length * d_model * batch_size: {(74.666 * num_layers + 8) * context_length * d_model * batch_size / 1e9:.2f}gb, {(74.666 * num_layers + 8) * context_length * d_model * batch_size / bytes_adamw:.2%}\n"
        f"as func of batch size: {16 * total_params/1e9:.2f}gb + {4 * total_activations_per_batch / 1e9:.2f}gb * batch_size")

    flop_MHA_attention = 2 * 2 * d_model * context_length**2 * num_layers
    flop_MHA_other = 2 * 4 * d_model**2 * context_length * num_layers
    flop_FF = (8/3) * d_model * d_model * context_length * 3 * num_layers * 2
    flop_final_lin = context_length * d_model * vocab_size * 2
    flop_total = flop_MHA_attention + flop_MHA_other + flop_FF+flop_final_lin
    print(f"\ntotal flop/(forward pass*batch): {flop_total : .2f}\n"
          f"total flop/(adamw*batch_size) = 3 * forward_pass: {3 * flop_total : .2f}\n"
          f"    MHA attention (d_model * context_len^2 * stuff): {flop_MHA_attention/flop_total:.2%} \n"
          f"    MHA other (d_model^2 * context_len * stuff): {flop_MHA_other/flop_total:.2%} \n"
          f"    FF:  {flop_FF/flop_total:.2%}\n"
          f"    Final layer:  {flop_final_lin/flop_total:.2%}\n")

    H100flop = 495e12
    util = 0.5
    # batch_size = 1024
    steps = 400000
    print(f"flop per train pass = 3 * batch_size * flop/(forward pass * batch) = {3} * {batch_size} * {flop_total:.2f} = {3 * batch_size * flop_total:.2f}\n"
          f"time per train pass = (flop per pass) / ((H100flop/s) * util)   =  {3 * batch_size * flop_total:.2f} / ({H100flop} * {util})  = {((3 * batch_size * flop_total)/(H100flop * util)):.2f}s\n"
          f"total time = (time per pass) * (steps) = {((3 * batch_size * flop_total)/(H100flop * util)):.2f}s * {steps} = {steps * ((3 * batch_size * flop_total)/(H100flop * util)) / (60*60):.2f}hr")




In [12]:
print("\nGPT2 XL ")

print("Adam accounting:")
adamw_accounting(vocab_size = 10000, 
            context_length = 256, 
            num_layers = 48, 
            d_model = 1600, 
            num_heads = 25,
            batch_size = 4)


GPT2 XL 
Adam accounting:
vocab_size: 10000
context_length: 256
num_layers: 48
d_model: 1600
num_heads: 25
batch_size: 4
total gb AdamW = 32.59gb = 
  (internal weights) 192 * d_model**2 * num_layers: 23.59gb, 72.40%
  (embedding weights) 32 * d_model * vocab_size: 0.51gb, 1.57%
  (logit type activations) 8 * context_length * vocab_size * batch_size: 0.08gb, 0.25%
  (attention activations)  8 * context_length**2 * num_heads * num_layers * batch_size: 2.52gb, 7.72%
  (internal activations) (74.666 * num_layers + 8) * context_length * d_model * batch_size: 5.89gb, 18.06%
as func of batch size: 24.10gb + 2.12gb * batch_size

total flop/(forward pass*batch):  783299379200.00
total flop/(adamw*batch_size) = 3 * forward_pass:  2349898137600.00
    MHA attention (d_model * context_len^2 * stuff): 2.57% 
    MHA other (d_model^2 * context_len * stuff): 32.13% 
    FF:  64.26%
    Final layer:  1.05%

flop per train pass = 3 * batch_size * flop/(forward pass * batch) = 3 * 4 * 783299379200.00 

In [13]:
print("My basic size")
adamw_accounting(vocab_size = 10000, 
            context_length = 256, 
            num_layers = 16, 
            d_model = 512, 
            num_heads = 4,
            batch_size = 4)

My basic size
vocab_size: 10000
context_length: 256
num_layers: 16
d_model: 512
num_heads: 4
batch_size: 4
total gb AdamW = 1.82gb = 
  (internal weights) 192 * d_model**2 * num_layers: 0.81gb, 44.35%
  (embedding weights) 32 * d_model * vocab_size: 0.16gb, 9.02%
  (logit type activations) 8 * context_length * vocab_size * batch_size: 0.08gb, 4.51%
  (attention activations)  8 * context_length**2 * num_heads * num_layers * batch_size: 0.13gb, 7.39%
  (internal activations) (74.666 * num_layers + 8) * context_length * d_model * batch_size: 0.63gb, 34.72%
as func of batch size: 0.97gb + 0.21gb * batch_size

total flop/(forward pass*batch):  30538727424.00
total flop/(adamw*batch_size) = 3 * forward_pass:  91616182272.00
    MHA attention (d_model * context_len^2 * stuff): 7.03% 
    MHA other (d_model^2 * context_len * stuff): 28.13% 
    FF:  56.26%
    Final layer:  8.58%

flop per train pass = 3 * batch_size * flop/(forward pass * batch) = 3 * 4 * 30538727424.00 = 366464729088.00
time

In [14]:
transformer_mem_and_flop(vocab_size = 10000, 
            context_length = 256, 
            num_layers = 16, 
            d_model = 512, 
            num_heads = 4)

vocab_size: 10000
context_length: 256
num_layers: 16
d_model: 512
num_heads: 4
d_ff: 1408

total params: 6.164e+07
    params embedding: 5.120e+06, 8.31%
    params MHA: 1.678e+07, 27.22%
    params FF: 3.460e+07, 56.14%
    params RMS: 1.690e+04, 0.03%
    params final lin: 5.120e+06, 8.31%
total gflop/forward pass:  31.08
    MHA attention (d_model * context_len^2 * stuff):  2.15, 6.91% 
    MHA other (d_model^2 * context_len * stuff): 8.59, 27.64% 
    FF: 17.72, 57.01%
    Final layer: 2.62, 8.44%

